In [26]:

import pandas as pd

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold


from sklearn.metrics import classification_report, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

In [27]:
pd.set_option('display.max_column',None)

In [28]:
df = pd.read_csv(r'C:\Users\mabin\Desktop\DataScienceClassNotes\Career247_Capstone_Project\Fraud_Detection_Project\data\raw\cleaned_data_v4.csv')

In [29]:
df.head()

,transaction_date,amount,ip_address_risk_score,device_trust_score,avg_amount_last_24h,device_change_flag,location_change_flag,merchant_historical_fraud_rate,cust_txn_count,cust_fraud_count,cust_fraud_rate,cust_avg_amt,device_count,merchant_fraud_rate,combined_risk,month,day,is_fraud
0,2024-01-01,556.63,0.041787,0.841375,3294.64,0,0,0.063596,11,0,0.0,9659.361818,11,0.073620,0.029911,1,1,0
1,2024-01-01,10158.89,0.162148,0.119578,4163.76,1,0,0.057505,11,0,0.0,9659.361818,11,0.099448,0.094997,1,1,0
2,2024-01-10,15754.57,0.774662,0.718447,5077.26,0,1,0.053729,11,0,0.0,9659.361818,11,0.077844,0.102428,1,10,0
3,2024-01-20,6095.68,0.259900,0.069586,1187.30,0,0,0.004334,11,0,0.0,9659.361818,11,0.086486,0.032338,1,20,0
4,2024-01-23,15324.24,0.376711,0.286630,10936.02,0,0,0.131218,11,0,0.0,9659.361818,11,0.096386,0.086691,1,23,0


In [30]:
numerical_col = df.select_dtypes(['int','float']).columns.to_list()

if 'is_fraud' in numerical_col:
    numerical_col.remove('is_fraud')

In [31]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),numerical_col)
    ],
    remainder='passthrough'
)

## Hyperparameter Tuning

In [32]:
param_grid = {
    'classifier__n_estimators': [200, 300, 500],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth': [3, 5, 7],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}

In [ ]:
def scorer(data, model):
    
    data = data.copy()
    
    # Convert to datetime
    data['transaction_date'] = pd.to_datetime(data['transaction_date'])
    
    
    # Divide the train and test data based on transaction date to remove Data Leakage
    train = data[data['transaction_date'] < '2024-03-01'].copy()
    test  = data[data['transaction_date'] >= '2024-03-01'].copy()
    
    train = train.drop(columns=['transaction_date'])
    test  = test.drop(columns=['transaction_date'])
    
    
    # Train test split
    X_train = train.drop('is_fraud', axis=1)
    y_train = train['is_fraud']

    X_test = test.drop('is_fraud', axis=1)
    y_test = test['is_fraud']

    
    # Correct pipeline with SMOTE inside
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', model)
    ])
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    search = GridSearchCV(pipeline, param_grid, scoring='recall', cv=skf, n_jobs=-1)
    
    
    search.fit(X_train, y_train)
    
    best_model = search.best_estimator_

    y_pred = best_model.predict(X_test)
    test_recall = recall_score(y_test, y_pred)
    
    return search.best_params_, float(search.best_score_), test_recall

In [34]:
result = scorer(df,GradientBoostingClassifier())
result

({'classifier__learning_rate': 0.01,
  'classifier__max_depth': 3,
  'classifier__min_samples_leaf': 1,
  'classifier__min_samples_split': 2,
  'classifier__n_estimators': 200},
 0.746895429331133,
 0.782608695652174)

The final model was tuned using hyperparameter optimization and achieved a cross-validation score of 0.7469 (74.69%).

The best-performing configuration was:

* Number of Estimators (n_estimators = 200)
The model uses 200 boosting stages, allowing it to learn complex patterns while maintaining stability.

* Learning Rate (learning_rate = 0.01)
A low learning rate ensures gradual learning, reducing the risk of overfitting and improving generalization.

* Max Depth (max_depth = 3)
Shallow trees were selected, indicating that simpler decision rules performed better and helped prevent overfitting.

* Minimum Samples Split (min_samples_split = 2)
Nodes are allowed to split when at least 2 samples are present, enabling flexibility in tree growth.

* Minimum Samples Leaf (min_samples_leaf = 1)
Leaf nodes can contain as few as 1 sample, allowing fine-grained decision boundaries.